# Fine-tune Generator (LoRA) — Tái lập Table 2 & Table 6

Notebook này fine-tune **Qwen1.5-14B-Chat** (đúng model trong paper) bằng LoRA trên dữ liệu `HIT-TMG/TruthReader_RAG_train`,
lưu adapter tại **mỗi epoch (1, 2, 3)** rồi đẩy lên HuggingFace Hub.

### Mapping với paper
| Artifact | Tái lập | Ghi chú |
|----------|---------|----------|
| LoRA 1 epoch | **Table 6** row Qwen (1 epoch) | Answer Acc, Refusal Rec, Citation Pre |
| LoRA 2 epochs | **Table 6** row Qwen (2 epochs) & **Table 2** row Adapted | Epoch tốt nhất theo paper |
| LoRA 3 epochs | **Table 6** row Qwen (3 epochs) | Kiểm tra overfitting |

### Giống hệt paper
- Base model: **Qwen/Qwen1.5-14B-Chat** — đúng model gốc trong paper (Table 2, Table 6)
- Training data: `HIT-TMG/TruthReader_RAG_train` (data đã xử lý bởi tác giả)
- LoRA: r=16, alpha=32, lr=1e-5, max_length=4096

### Yêu cầu GPU

- **A100 40GB** hoặc lớn hơn (14B params bfloat16 ≈ 28GB + LoRA overhead)3 model sẽ có trên HuggingFace, dùng cho notebook **TruthReader_Reproduce.ipynb** để đánh giá.

### Sau khi chạy xong

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'modal'])
import modal
print('modal:', modal.__version__)

In [ ]:
# ╔══════════════════════════════════════════╗
# ║          CẤU HÌNH — SỬA Ở ĐÂY           ║
# ╚══════════════════════════════════════════╝

HF_TOKEN    = "hf_YOUR_TOKEN_HERE"       # HuggingFace write token
HF_USERNAME = "trinhtrantran122"          # HuggingFace username

BASE_MODEL   = "Qwen/Qwen1.5-14B-Chat"     # Đúng model trong paper
TRAIN_DATASET = "HIT-TMG/TruthReader_RAG_train"  # Data đã xử lý bởi tác giả

# HuggingFace repo cho từng epoch
HF_REPOS = {
    1: f"{HF_USERNAME}/qwen1.5-14b-truthreader-1ep",
    2: f"{HF_USERNAME}/qwen1.5-14b-truthreader-2ep",
    3: f"{HF_USERNAME}/qwen1.5-14b-truthreader-3ep",
}

MAX_EPOCHS = 3   # Train 3 epochs, save adapter at each
print('Repos:', HF_REPOS)

In [ ]:
import modal

image = (
    modal.Image.debian_slim(python_version='3.11')
    .pip_install(
        'torch', 'transformers>=4.44.0', 'peft>=0.12.0', 'trl>=0.9.0',
        'datasets', 'accelerate', 'bitsandbytes',
        'huggingface_hub', 'langdetect',
    )
)

app = modal.App('truthreader-finetune-gen', image=image)
vol = modal.Volume.from_name('truthreader-ft-gen', create_if_missing=True)
VOL = '/vol'

print('Modal app defined')

## Step 1: Download data & chuyển sang SFT format

In [ ]:
@app.function(volumes={VOL: vol}, timeout=1800)
def prepare_sft_data(dataset_id: str):
    """Download HIT-TMG data, convert to SFT messages format."""
    import json, os, random
    from datasets import load_dataset
    random.seed(42)

    out_dir = os.path.join(VOL, 'sft')
    train_path = os.path.join(out_dir, 'train.jsonl')
    val_path   = os.path.join(out_dir, 'val.jsonl')

    if os.path.exists(train_path):
        n = sum(1 for _ in open(train_path))
        print(f'SFT data already exists: {n} train examples')
        vol.commit()
        return {'train': n, 'status': 'cached'}

    print(f'Downloading {dataset_id}...')
    ds = load_dataset(dataset_id)
    all_items = []
    for split in ds:
        for item in ds[split]:
            all_items.append(item)
    print(f'Total raw examples: {len(all_items)}')

    def detect_lang(text):
        from langdetect import detect, DetectorFactory
        DetectorFactory.seed = 0
        try:
            l = detect(text[:300])
            return 'zh' if l.startswith('zh') else l
        except Exception:
            return 'en'

    def convert(ex):
        docs = ex.get('documents', [])
        q    = ex.get('question', '')
        ans  = ex.get('answer', '')
        hist = ex.get('history', [])
        doc_parts = [f'## Document[{i+1}]\t{d.get("title","")}\n{d.get("document","")}'
                     for i, d in enumerate(docs)]
        docs_str = '\n\n'.join(doc_parts)

        lang = detect_lang(q)
        if lang == 'zh':
            sys_msg = '请基于给定的文档，生成问题的答案。如果文档中没有包含答案的信息，请回复抱歉并给出理由。'
        elif lang == 'vi':
            sys_msg = 'Dựa trên các tài liệu được cung cấp, hãy tạo câu trả lời cho câu hỏi. Nếu tài liệu không chứa thông tin để trả lời câu hỏi, hãy xin lỗi và giải thích lý do.'
        else:
            sys_msg = 'Based on the given documents, generate an answer to the question. If the documents do not contain information to answer the question, please reply with an apology and explain the reason.'

        user = f'{sys_msg}\n\n# DOCUMENTS:\n{docs_str}\n\n# QUESTION: {q}\n\n# ANSWER: '
        msgs = [{'role': 'system', 'content': 'You are a helpful assistant.'}]
        if hist and len(hist) >= 2:
            for i in range(0, len(hist) - 1, 2):
                msgs.append({'role': 'user', 'content': hist[i]})
                msgs.append({'role': 'assistant', 'content': hist[i+1]})
        msgs.append({'role': 'user', 'content': user})
        msgs.append({'role': 'assistant', 'content': ans})
        return {'messages': msgs}

    converted = [convert(ex) for ex in all_items if ex.get('answer', '').strip()]
    random.shuffle(converted)
    val_n = max(1, int(len(converted) * 0.05))
    val_data, train_data = converted[:val_n], converted[val_n:]

    os.makedirs(out_dir, exist_ok=True)
    for path, data in [(train_path, train_data), (val_path, val_data)]:
        with open(path, 'w', encoding='utf-8') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

    vol.commit()
    print(f'Train: {len(train_data)},  Val: {len(val_data)}')
    return {'train': len(train_data), 'val': len(val_data), 'status': 'created'}


with app.run():
    sft_info = prepare_sft_data.remote(TRAIN_DATASET)
print(sft_info)

## Step 2: LoRA Fine-tune (3 epochs, lưu adapter mỗi epoch)

Paper Section 4.2.2: LoRA, lr=1e-5, max_length=4096, negative log likelihood loss.

Chạy **1 lần**, tự động lưu checkpoint tại epoch 1, 2, 3.

In [ ]:
@app.function(
    volumes={VOL: vol},
    gpu='A100',
    timeout=36000,  # 10h
)
def lora_finetune(base_model: str, max_epochs: int):
    import os, json, glob, torch
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, TaskType
    from trl import SFTTrainer, SFTConfig

    sft_dir = os.path.join(VOL, 'sft')
    output_dir = os.path.join(VOL, 'generator_checkpoints')

    print(f'=== LoRA Fine-tune: {base_model}, {max_epochs} epochs ===')
    print(f'GPU: {torch.cuda.get_device_name(0)}')

    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
        device_map='auto',
        trust_remote_code=True,
    )
    model.config.use_cache = False

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        bias='none',
    )

    dataset = load_dataset('json', data_files={
        'train': os.path.join(sft_dir, 'train.jsonl'),
        'validation': os.path.join(sft_dir, 'val.jsonl'),
    })
    print(f'Train: {len(dataset["train"])}, Val: {len(dataset["validation"])}')

    training_args = SFTConfig(
        output_dir=output_dir,
        num_train_epochs=max_epochs,
        learning_rate=1e-5,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_ratio=0.05,
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        logging_steps=10,
        save_strategy='epoch',
        eval_strategy='epoch',
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        max_seq_length=4096,
        gradient_checkpointing=True,
        report_to='none',
        seed=42,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset['train'],
        eval_dataset=dataset['validation'],
        processing_class=tokenizer,
        peft_config=lora_config,
    )

    print('\nStarting training...')
    result = trainer.train()
    print(f'Training done. Loss: {result.metrics.get("train_loss", "?")}')

    # Lưu tokenizer vào mỗi checkpoint (cần cho loading)
    checkpoints = sorted(glob.glob(os.path.join(output_dir, 'checkpoint-*')))
    for ckpt in checkpoints:
        tokenizer.save_pretrained(ckpt)

    # Cũng lưu adapter cuối cùng (epoch 3)
    final_dir = os.path.join(output_dir, 'final')
    trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)

    vol.commit()

    info = {
        'train_loss': result.metrics.get('train_loss'),
        'runtime_s': result.metrics.get('train_runtime'),
        'checkpoints': [os.path.basename(c) for c in checkpoints],
    }
    print(f'Checkpoints: {info["checkpoints"]}')
    return info


with app.run():
    train_info = lora_finetune.remote(BASE_MODEL, MAX_EPOCHS)

print('\n=== Training Complete ===')
print(f'Loss: {train_info["train_loss"]}')
print(f'Checkpoints: {train_info["checkpoints"]}')

## Step 3: Đẩy 3 adapter lên HuggingFace Hub

Mỗi checkpoint (epoch 1, 2, 3) → 1 repo riêng trên HF.

In [ ]:
@app.function(
    volumes={VOL: vol},
    timeout=3600,
    secrets=[modal.Secret.from_dict({'HF_TOKEN': HF_TOKEN})],
)
def push_to_hf(hf_repos: dict, base_model: str):
    """Push each epoch checkpoint to its own HF repo."""
    import os, glob, json
    from huggingface_hub import HfApi, create_repo
    from pathlib import Path

    token = os.environ['HF_TOKEN']
    api = HfApi(token=token)

    ckpt_dir = os.path.join(VOL, 'generator_checkpoints')
    checkpoints = sorted(glob.glob(os.path.join(ckpt_dir, 'checkpoint-*')))
    print(f'Found {len(checkpoints)} checkpoints: {[os.path.basename(c) for c in checkpoints]}')

    results = {}
    for epoch_num, repo_id in sorted(hf_repos.items()):
        idx = epoch_num - 1
        if idx >= len(checkpoints):
            # Dùng final nếu không đủ checkpoint
            ckpt_path = os.path.join(ckpt_dir, 'final')
        else:
            ckpt_path = checkpoints[idx]

        if not os.path.exists(ckpt_path):
            print(f'SKIP epoch {epoch_num}: {ckpt_path} not found')
            continue

        # Tạo README model card
        readme = f"""# TruthReader Generator — LoRA Adapter (Epoch {epoch_num})

Fine-tuned for RAG with inline citations (TruthReader paper reproduction).

- **Base model**: `{base_model}`
- **Training data**: `HIT-TMG/TruthReader_RAG_train`
- **Epochs**: {epoch_num} / 3
- **LoRA**: r=16, alpha=32
- **Learning rate**: 1e-5
- **Max seq length**: 4096

## Usage
```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained("{base_model}", device_map="auto")
model = PeftModel.from_pretrained(base, "{repo_id}")
```

## Paper
TruthReader: Towards Trustworthy Document Assistant Chatbot with Reliable Attribution (EMNLP 2024)
"""
        readme_path = os.path.join(ckpt_path, 'README.md')
        with open(readme_path, 'w') as f:
            f.write(readme)

        print(f'\nPushing epoch {epoch_num} → {repo_id}')
        create_repo(repo_id=repo_id, token=token, private=True,
                    repo_type='model', exist_ok=True)
        api.upload_folder(
            folder_path=ckpt_path,
            repo_id=repo_id,
            repo_type='model',
            commit_message=f'Upload LoRA adapter epoch {epoch_num}',
        )
        results[epoch_num] = f'https://huggingface.co/{repo_id}'
        print(f'OK: {results[epoch_num]}')

    return results


with app.run():
    hf_urls = push_to_hf.remote(HF_REPOS, BASE_MODEL)

print('\n=== HuggingFace Upload Complete ===')
for ep, url in hf_urls.items():
    print(f'  Epoch {ep}: {url}')

## Done!

3 LoRA adapter đã được đẩy lên HuggingFace:

| Epoch | HF Repo | Dùng cho |
|-------|---------|----------|
| 1 | `{HF_USERNAME}/qwen2.5-7b-truthreader-1ep` | Table 6 row 1 |
| 2 | `{HF_USERNAME}/qwen2.5-7b-truthreader-2ep` | Table 6 row 2, Table 2 |
| 3 | `{HF_USERNAME}/qwen2.5-7b-truthreader-3ep` | Table 6 row 3 |

→ Tiếp theo: chạy **TruthReader_Reproduce.ipynb** để đánh giá.